# 02 — Silver Layer Cleaning, Standardization, Data Quality & Quarantine
- **Workstream 2:** Lakehouse & Data Quality (Member 2 — Ahmed Essam)
- **ClickUp Tasks:**
  - `123u8tync4v` (*Implement Silver cleaning and standardization*)
  - `123u8tync4x` (*Create data-quality checks and report*)
- **Key Rule Enforced:** Quarantine bad/conflicting records with explicit reason codes (`silver_quarantine_records`); never silently drop records.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_ndc = spark.table("workspace.default.bronze_openfda_ndc")

# 1. Deduplication check via Window over product_id
w_pk = Window.partitionBy("product_id").orderBy(F.col("ingestion_timestamp_utc").desc())
staged_ndc = (
    bronze_ndc
    .withColumn("row_num", F.row_number().over(w_pk))
    .withColumn("has_active_ingredients", F.expr("active_ingredients IS NOT NULL AND size(active_ingredients) > 0 AND active_ingredients[0].name IS NOT NULL AND active_ingredients[0].strength IS NOT NULL"))
    .withColumn("has_primary_key", F.expr("product_id IS NOT NULL AND trim(product_id) != '' AND product_ndc IS NOT NULL"))
    .withColumn("has_drug_name", F.expr("(brand_name IS NOT NULL AND trim(brand_name) != '') OR (generic_name IS NOT NULL AND trim(generic_name) != '')"))
)

# 2. Split into Valid Silver vs Quarantine with explicit reason codes
quarantine_ndc = (
    staged_ndc.filter("NOT (row_num = 1 AND has_active_ingredients AND has_primary_key AND has_drug_name)")
    .select(
        F.concat(F.lit("Q-NDC-"), F.monotonically_increasing_id()).alias("quarantine_id"),
        F.lit("bronze_openfda_ndc").alias("source_table"),
        F.coalesce(F.col("product_id"), F.col("product_ndc"), F.lit("UNKNOWN")).alias("record_identifier"),
        F.when(~F.col("has_primary_key"), F.lit("MISSING_PRIMARY_IDENTIFIER"))
         .when(F.col("row_num") > 1, F.lit("DUPLICATE_PRIMARY_KEY"))
         .when(~F.col("has_drug_name"), F.lit("MISSING_DRUG_NAME"))
         .otherwise(F.lit("MISSING_ACTIVE_INGREDIENT_OR_STRENGTH")).alias("quarantine_reason_code"),
        F.lit("QUARANTINED_CRITICAL").alias("severity"),
        F.to_json(F.struct("*")).alias("raw_payload_json"),
        F.current_timestamp().alias("quarantined_at_utc")
    )
)

silver_ndc = (
    staged_ndc.filter("row_num = 1 AND has_active_ingredients AND has_primary_key AND has_drug_name")
    .select(
        F.trim(F.col("product_id")).alias("product_id"),
        F.trim(F.col("product_ndc")).alias("product_ndc"),
        F.upper(F.coalesce(F.col("brand_name"), F.col("generic_name"))).alias("brand_name"),
        F.upper(F.coalesce(F.col("generic_name"), F.col("brand_name"))).alias("generic_name"),
        F.upper(F.coalesce(F.col("labeler_name"), F.lit("UNKNOWN LABELER"))).alias("labeler_name"),
        F.upper(F.col("active_ingredients")[0]["name"]).alias("primary_active_ingredient"),
        F.col("active_ingredients")[0]["strength"].alias("primary_strength"),
        F.size(F.col("active_ingredients")).alias("ingredient_count"),
        F.upper(F.coalesce(F.col("dosage_form"), F.lit("UNKNOWN"))).alias("dosage_form"),
        F.upper(F.coalesce(F.col("product_type"), F.lit("UNKNOWN"))).alias("product_type"),
        F.upper(F.coalesce(F.col("marketing_category"), F.lit("UNSPECIFIED"))).alias("marketing_category"),
        F.size(F.col("packaging")).alias("package_count"),
        F.to_date(F.col("marketing_start_date"), "yyyyMMdd").alias("marketing_start_date_iso"),
        F.to_date(F.col("listing_expiration_date"), "yyyyMMdd").alias("listing_expiration_date_iso"),
        "source_id",
        "raw_record_sha256"
    )
)

silver_ndc.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_openfda_ndc")
quarantine_ndc.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_quarantine_records")

print(f"Silver NDC valid rows: {silver_ndc.count()} | Quarantined rows: {quarantine_ndc.count()}")
display(quarantine_ndc)

In [ ]:
# 3. Clean & Standardize openFDA Enforcement Recalls
bronze_enf = spark.table("workspace.default.bronze_openfda_enforcement")
w_enf = Window.partitionBy("recall_number").orderBy(F.col("ingestion_timestamp_utc").desc())

silver_enf = (
    bronze_enf
    .withColumn("row_num", F.row_number().over(w_enf))
    .filter("row_num = 1 AND recall_number IS NOT NULL AND reason_for_recall IS NOT NULL")
    .select(
        F.trim(F.col("recall_number")).alias("recall_number"),
        F.col("event_id"),
        F.col("classification"),
        F.when(F.col("classification") == "Class I", 1)
         .when(F.col("classification") == "Class II", 2)
         .when(F.col("classification") == "Class III", 3)
         .otherwise(4).alias("severity_rank"),
        F.initcap(F.col("status")).alias("status"),
        F.upper(F.col("recalling_firm")).alias("recalling_firm"),
        F.col("city"),
        F.col("state"),
        F.upper(F.coalesce(F.col("country"), F.lit("UNITED STATES"))).alias("country"),
        F.when(F.lower(F.col("reason_for_recall")).rlike("steril"), "STERILITY_FAILURE")
         .when(F.lower(F.col("reason_for_recall")).rlike("microbial|contaminat|methanol|impurity|ndma|particulate"), "CONTAMINATION_OR_IMPURITY")
         .when(F.lower(F.col("reason_for_recall")).rlike("cgmp|good manufacturing"), "CGMP_DEVIATION")
         .when(F.lower(F.col("reason_for_recall")).rlike("unapproved|new drug"), "UNAPPROVED_DRUG_MARKETING")
         .when(F.lower(F.col("reason_for_recall")).rlike("label|misbrand|packaging"), "LABELING_OR_PACKAGING_DEFECT")
         .when(F.lower(F.col("reason_for_recall")).rlike("subpotent|superpotent|potency|dissolution|stability"), "POTENCY_OR_STABILITY_FAILURE")
         .otherwise("OTHER_QUALITY_DEFECT").alias("recall_reason_category"),
        F.col("reason_for_recall"),
        F.col("product_description"),
        F.to_date(F.col("recall_initiation_date"), "yyyyMMdd").alias("recall_initiation_date_iso"),
        F.to_date(F.col("report_date"), "yyyyMMdd").alias("report_date_iso"),
        "source_id",
        "raw_record_sha256"
    )
)

silver_enf.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_openfda_enforcement")
print(f"Silver Enforcement valid rows: {silver_enf.count()}")
display(silver_enf.limit(5))